# 02 Facets Proposals

Computes proposal-side diversity facets M0-M6 over `condition × text_version`, using only prepared artifacts from `4a` and the frozen literature map. Metrics are computed in full embedding space except M6, which consumes the abstract-derived proposal-to-literature KNN cache. Figures are regenerated from tidy output tables/curves, not from metric objects.

In [1]:
CONFIG = dict(
    conditions=["baseline", "one_at_a_time", "persona"],
    text_versions=["rephrased", "original"],
    fields=["whole"],
    models=["claude", "gemini", "gpt"],
    n_human=23,
    seed=42,
    B_perm=10_000,
    B_sub=1_000,
    m6_required=True,
    literature_ks=[5, 10, 20],
)

RUN_OT = True  # set False only if POT is unavailable; MMD2 remains the primary M5 statistic
WRITE_FIGURES = True

## Load

Load proposal prep artifacts and assert the prep-to-analysis contract before any positional cache is used.

In [2]:
import json
import pickle
import sys
from dataclasses import replace
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import diversity_inference as di
import plotting as pl
from proposal_comparison import load_condition_analysis_inputs
from prepare_proposals_for_analysis import load_pickle

if not RUN_OT:
    di.wasserstein_ot = lambda X, Y: np.nan

TEST_COLUMNS = [
    "condition", "task", "text_version", "field", "comparison", "facet", "metric", "is_primary", "param",
    "human_value", "ai_value", "effect_size", "effect_type", "ci_lo", "ci_hi", "human_ci_lo", "human_ci_hi",
    "inference", "stat", "p_raw", "p_fdr", "n_human", "n_ai", "n_perm_or_sub", "parity_ref", "notes",
]
# Pre-registered primaries per spec 1.7 + the facet-primary flags of spec 12.1.
# M6 metrics are secondary/convergent (spec 1.7) - NOT primary.
PRIMARY = {
    ("spread", "mean_pairwise", ""),
    ("richness", "vendi", "q=1"),
    ("coverage", "coverage_geometric", "k=3"),
    ("dimensionality", "participation_ratio", ""),
    ("evenness", "ripley_excess", "r=pooled_q01_q50"),
    ("displacement", "mmd2", ""),
}

def _standardize_tests(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    if "n_perm_or_boot" in out.columns:
        out = out.rename(columns={"n_perm_or_boot": "n_perm_or_sub"})
    for col in TEST_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan if col not in {"field", "param", "notes"} else ""
    out["param"] = out["param"].fillna("")
    out["is_primary"] = out.apply(lambda r: (r["facet"], r["metric"], r["param"]) in PRIMARY, axis=1)
    out.loc[out["facet"].ne("coverage"), "parity_ref"] = 1.0
    coverage_mask = out["metric"].eq("coverage_geometric")
    out.loc[coverage_mask, "parity_ref"] = out.loc[coverage_mask, "human_value"]
    domain_mask = out["metric"].isin(["coverage_bertopic_region", "coverage_mesh_terms"])
    out.loc[domain_mask, "parity_ref"] = 1.0
    return out[TEST_COLUMNS]

def _standardize_gradient(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"JT", "p_raw", "p_fdr"} else ""
    return out[["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]]

def _standardize_curves(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "task", "text_version", "field", "group", "facet", "metric", "param", "x", "y", "y_lo", "y_hi"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"x", "y", "y_lo", "y_hi"} else ""
    out["param"] = out["param"].fillna("")
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    return out[["condition", "task", "text_version", "field", "group", "facet", "metric", "param", "x", "y", "y_lo", "y_hi"]]

def _load_json(path):
    return json.loads(Path(path).read_text())

def _assert_proposal_contract(condition, text_version):
    prep_dir = PROJECT_ROOT / "data" / "prepared" / condition / "proposals" / text_version
    manifest = _load_json(prep_dir / "prepare_manifest.json")
    master = pd.read_csv(prep_dir / "proposal_master.csv")
    assert manifest["embeddings_l2_normalized"] is True, f"run modified 4a first: {condition}/{text_version}"
    assert manifest["proposal_uid_order"] == master["proposal_uid"].astype(str).tolist(), "row order drift in proposal master"
    assert "model" in master.columns, "MODEL_COL missing: expected proposal_master['model']"
    ai_counts = master.loc[master["source_type"].eq("ai"), "source_group"].value_counts().to_dict()
    assert all(ai_counts.get(g, 0) == CONFIG["n_human"] for g in ["Claude", "Gemini", "GPT"]), ai_counts
    idx_path = Path(manifest["subsample_idx_file"])
    idx_cache = np.load(idx_path)
    assert idx_cache.shape == (CONFIG["B_sub"], CONFIG["n_human"]), f"bad subsample shape: {idx_cache.shape}"
    return prep_dir, manifest, master, idx_cache

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal


## M0-M5 Proposal Facets

Compute spread, richness, geometric coverage, dimensionality, evenness, and displacement using full-text proposal embeddings.

In [3]:
all_tests = []
all_gradients = []
all_curves = []
all_nulls = []

NULL_COLUMNS = ["condition", "task", "text_version", "field", "facet", "metric", "param", "draw_idx", "value"]

def _standardize_nulls(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    out["param"] = out["param"].fillna("")
    return out[NULL_COLUMNS]

for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        print(f"\n=== Proposals: {condition}/{text_version} ===")
        prep_dir, manifest, master, idx_cache = _assert_proposal_contract(condition, text_version)
        analysis = load_condition_analysis_inputs(PROJECT_ROOT, condition, text_version=text_version)
        tests, gradients, curves, nulls = di.build_proposal_facet_outputs(
            analysis,
            text_branch=text_version,
            bootstrap_ai_idx_samples=idx_cache,
            n_perm=CONFIG["B_perm"],
            n_boot=CONFIG["B_sub"],
            seed=CONFIG["seed"],
        )
        all_tests.append(_standardize_tests(tests))
        all_gradients.append(_standardize_gradient(gradients))
        all_curves.append(_standardize_curves(curves))
        all_nulls.append(_standardize_nulls(nulls))

proposal_tests_df = pd.concat(all_tests, ignore_index=True) if all_tests else pd.DataFrame(columns=TEST_COLUMNS)
proposal_gradient_df = pd.concat(all_gradients, ignore_index=True) if all_gradients else pd.DataFrame()
proposal_curves_df = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
proposal_nulls_df = pd.concat(all_nulls, ignore_index=True) if all_nulls else pd.DataFrame(columns=NULL_COLUMNS)
proposal_tests_df.head()


=== Proposals: baseline/rephrased ===



=== Proposals: baseline/original ===



=== Proposals: one_at_a_time/rephrased ===



=== Proposals: one_at_a_time/original ===



=== Proposals: persona/rephrased ===



=== Proposals: persona/original ===


,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,human_ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
0,baseline,proposals,rephrased,whole,human_vs_claude,spread,mean_pairwise,True,,0.415570,...,0.424403,permutation,0.160459,0.079292,NaN,23,23,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
1,baseline,proposals,rephrased,whole,human_vs_claude,spread,centroid_loo,False,,0.640121,...,0.651821,permutation,0.217616,0.046395,NaN,23,23,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
2,baseline,proposals,rephrased,whole,human_vs_claude,spread,mst_dispersion,False,,0.110074,...,0.114508,permutation,0.030784,0.066293,NaN,23,23,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
3,baseline,proposals,rephrased,whole,human_vs_claude,spread,sparseness,False,,0.322715,...,0.336489,permutation,0.163548,0.062494,NaN,23,23,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."
4,baseline,proposals,rephrased,whole,human_vs_claude,spread,nn_isolation,False,,0.081982,...,0.085081,permutation,0.028900,0.073293,NaN,23,23,10000.0,1.0,"spread facet; ci = 95% jackknife (AI group), h..."


## M6 Coverage Domain

Consume the frozen literature map, proposal-to-literature KNN cache, and MeSH index. No literature model is refit here.

In [4]:
def _load_literature_domain_assets():
    lit_dir = PROJECT_ROOT / "data" / "prepared" / "literature"
    lit_manifest = _load_json(lit_dir / "literature_prepare_manifest.json")
    if CONFIG["m6_required"]:
        assert lit_manifest["bertopic_status"] == "ready", "M6 requires ready BERTopic from 4a"
        assert lit_manifest["mesh_available"] is True, "M6 requires MeSH index from 4a"
    assignments = pd.read_csv(lit_manifest["bertopic_assignments_file"]).sort_values("article_idx")
    assert (assignments["article_idx"].to_numpy() == np.arange(len(assignments))).all(), "non-contiguous article_idx"
    topic_by_row = assignments["bertopic_topic"].astype(int).to_numpy()
    mesh_df = pd.read_parquet(lit_manifest["mesh_index_file"]).sort_values("article_row_idx")
    assert (mesh_df["article_row_idx"].to_numpy() == np.arange(len(mesh_df))).all(), "non-contiguous mesh index"
    mesh_by_row = [list(v) if isinstance(v, (list, tuple, np.ndarray)) else [] for v in mesh_df["mesh_terms"].tolist()]
    n_regions = int(lit_manifest.get("n_regions_excl_outlier", 0))
    topic_info_path = lit_dir / "lit_bertopic_topic_info.csv"
    topic_labels = {}
    if topic_info_path.exists():
        info = pd.read_csv(topic_info_path)
        name_col = "Name" if "Name" in info.columns else ("name" if "name" in info.columns else None)
        id_col = "Topic" if "Topic" in info.columns else ("topic" if "topic" in info.columns else None)
        if name_col and id_col:
            topic_labels = {int(t): str(n)[:40] for t, n in zip(info[id_col], info[name_col])}
    return topic_by_row, mesh_by_row, n_regions, topic_labels


def _build_row_masks(knn_payload, topic_by_row, mesh_by_row, k_lit):
    """Per-proposal bitmasks: which regions / MeSH terms its k nearest abstracts touch.

    Bitmasks make union counts (the M6 statistic) cheap enough for a B=10,000
    label-permutation test (spec 9.3) instead of the placeholder inference.
    """
    nn = np.asarray(knn_payload["neighbor_idx"])[:, :k_lit]
    topic_masks, mesh_masks = [], []
    mesh_vocab = {}
    for row in nn:
        t_mask = 0
        m_mask = 0
        for i in row:
            t = int(topic_by_row[int(i)])
            if t != -1:
                t_mask |= 1 << t
            for term in mesh_by_row[int(i)]:
                term = str(term).strip()
                if not term:
                    continue
                pos = mesh_vocab.setdefault(term, len(mesh_vocab))
                m_mask |= 1 << pos
        topic_masks.append(t_mask)
        mesh_masks.append(m_mask)
    return topic_masks, mesh_masks


def _m6_rows_for_cell(condition, text_version, master, idx_cache):
    prep_dir = PROJECT_ROOT / "data" / "prepared" / condition / "proposals" / text_version
    knn = np.load(prep_dir / "proposal_to_literature_knn.npz", allow_pickle=True)
    human_rows = master.index[master["source_type"].eq("human")].to_numpy()
    ai_rows = master.index[master["source_type"].eq("ai")].to_numpy()
    group_rows = {g: master.index[master["source_group"].eq(g)].to_numpy() for g in ["Claude", "Gemini", "GPT"]}
    rows, curves, grads, nulls = [], [], [], []
    rng = np.random.default_rng(CONFIG["seed"])
    note = "M6 uses abstract-derived KNN; BERTopic outlier bin (-1) excluded from unions"

    for k_lit in CONFIG["literature_ks"]:
        topic_masks, mesh_masks = _build_row_masks(knn, TOPIC_BY_ROW, MESH_BY_ROW, k_lit)
        param = f"k_lit={k_lit}"
        h_jk = {"coverage_bertopic_region": di.jackknife_union(topic_masks, human_rows),
                "coverage_mesh_terms": di.jackknife_union(mesh_masks, human_rows)}

        # Per-model 23-vs-23: label permutation on the union-count difference (spec 9.3).
        for group, g_rows in group_rows.items():
            comparison = f"human_vs_{group.lower()}"
            for metric, masks in [("coverage_bertopic_region", topic_masks), ("coverage_mesh_terms", mesh_masks)]:
                perm = di.permutation_union_test(masks, human_rows, g_rows, B=CONFIG["B_perm"], seed=CONFIG["seed"])
                hj = h_jk[metric]
                gj = di.jackknife_union(masks, g_rows)
                rows.append({
                    "condition": condition, "task": "proposals", "text_version": text_version, "field": "whole",
                    "comparison": comparison, "facet": "coverage", "metric": metric, "is_primary": False,
                    "param": param, "human_value": hj["point"], "ai_value": gj["point"],
                    "effect_size": hj["point"] / gj["point"] if gj["point"] else np.nan, "effect_type": "ratio",
                    "ci_lo": gj["lo"], "ci_hi": gj["hi"], "human_ci_lo": hj["lo"], "human_ci_hi": hj["hi"],
                    "inference": "permutation", "stat": perm["delta_obs"], "p_raw": perm["p_two_sided"], "p_fdr": np.nan,
                    "n_human": CONFIG["n_human"], "n_ai": CONFIG["n_human"], "n_perm_or_sub": CONFIG["B_perm"],
                    "parity_ref": 1.0,
                    "notes": note + "; equal-n union comparison; ci = 95% jackknife",
                })

        # Pooled Human-vs-All-AI: MUST go through the subsample path (spec 9.3 - union
        # metrics are the most n-sensitive in the spec; never 23-vs-69 raw).
        for metric, masks in [("coverage_bertopic_region", topic_masks), ("coverage_mesh_terms", mesh_masks)]:
            hj = h_jk[metric]
            vals = np.asarray([di.union_count(masks, sample) for sample in idx_cache], dtype=float)
            ai_mean = float(np.mean(vals))
            p = float((np.sum(vals >= hj["point"]) + 1) / (len(vals) + 1))
            rows.append({
                "condition": condition, "task": "proposals", "text_version": text_version, "field": "whole",
                "comparison": "human_vs_pooled_ai", "facet": "coverage", "metric": metric, "is_primary": False,
                "param": param, "human_value": hj["point"], "ai_value": ai_mean,
                "effect_size": hj["point"] / ai_mean if ai_mean else np.nan, "effect_type": "ratio",
                "ci_lo": float(np.percentile(vals, 2.5)), "ci_hi": float(np.percentile(vals, 97.5)),
                "human_ci_lo": hj["lo"], "human_ci_hi": hj["hi"],
                "inference": "same_size_subsample", "stat": hj["point"] - ai_mean, "p_raw": p, "p_fdr": np.nan,
                "n_human": CONFIG["n_human"], "n_ai": CONFIG["n_human"], "n_perm_or_sub": len(vals),
                "parity_ref": 1.0,
                "notes": note + "; AI n=23 subsampled from 69 (1000 draws, without replacement)",
            })

        # Gradient (spec 9.3): JT on per-model unions at matched n, over jackknife replicates.
        for metric, masks in [("coverage_bertopic_region", topic_masks), ("coverage_mesh_terms", mesh_masks)]:
            reps = [di.jackknife_union(masks, group_rows[g])["replicates"] for g in ["Claude", "Gemini", "GPT"]]
            reps.append(h_jk[metric]["replicates"])
            points = [di.union_count(masks, group_rows[g]) for g in ["Claude", "Gemini", "GPT"]] + [h_jk[metric]["point"]]
            jt = di.jonckheere_terpstra(reps, alternative="increasing")
            grads.append({"condition": condition, "task": "proposals", "text_version": text_version, "field": "whole",
                          "facet": "coverage", "metric": metric, "param": param, "order": "claude<gemini<gpt<human",
                          "JT": jt["JT"], "p_raw": jt["p"], "p_fdr": np.nan,
                          "direction_ok": bool(all(x <= y for x, y in zip(points, points[1:]))),
                          "notes": "JT over LOO jackknife replicates of the union count at matched n"})

        # Rarefaction curves (primary M6 presentation, spec 9.3).
        for group, rows_for_group in {"Human": human_rows, **group_rows}.items():
            for m in range(1, CONFIG["n_human"] + 1):
                draws = [rng.choice(rows_for_group, size=m, replace=False) for _ in range(min(200, CONFIG["B_sub"]))]
                reg_vals = [di.union_count(topic_masks, d) for d in draws]
                mesh_vals = [di.union_count(mesh_masks, d) for d in draws]
                for metric, vals in [("coverage_bertopic_region_rarefaction", reg_vals), ("coverage_mesh_terms_rarefaction", mesh_vals)]:
                    curves.append({"condition": condition, "task": "proposals", "text_version": text_version,
                                   "field": "whole", "group": group, "facet": "coverage", "metric": metric,
                                   "param": param, "x": m, "y": float(np.mean(vals)),
                                   "y_lo": float(np.percentile(vals, 2.5)), "y_hi": float(np.percentile(vals, 97.5))})

        if k_lit == 10:
            # Region-occupancy matrix (spec 9.4.2): proposals per region per group.
            for group, rows_for_group in {"Human": human_rows, **group_rows}.items():
                for region in range(N_REGIONS_TOTAL):
                    count = sum(1 for r in rows_for_group if (topic_masks[int(r)] >> region) & 1)
                    curves.append({"condition": condition, "task": "proposals", "text_version": text_version,
                                   "field": "whole", "group": group, "facet": "coverage",
                                   "metric": "region_occupancy", "param": param, "x": region, "y": float(count),
                                   "y_lo": np.nan, "y_hi": np.nan})
            # Fingerprint null draws (redesign spec 3.1): union counts over M=999 same-n
            # draws of the pooled proposal cloud; dedicated rng stream so nothing else moves.
            rng_null = np.random.default_rng(CONFIG["seed"] + 2)
            all_rows = master.index.to_numpy()
            for i in range(999):
                draw = rng_null.choice(all_rows, size=CONFIG["n_human"], replace=False)
                nulls.append({"condition": condition, "task": "proposals", "text_version": text_version,
                              "field": "whole", "facet": "coverage", "metric": "coverage_bertopic_region",
                              "param": param, "draw_idx": i, "value": float(di.union_count(topic_masks, draw))})
    return pd.DataFrame(rows), pd.DataFrame(curves), pd.DataFrame(grads), pd.DataFrame(nulls)


TOPIC_BY_ROW, MESH_BY_ROW, N_REGIONS_TOTAL, TOPIC_LABELS = _load_literature_domain_assets()
m6_tests, m6_curves, m6_grads, m6_nulls = [], [], [], []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        _, _, master, idx_cache = _assert_proposal_contract(condition, text_version)
        t, c, g, nl = _m6_rows_for_cell(condition, text_version, master, idx_cache)
        m6_tests.append(t)
        m6_curves.append(c)
        m6_grads.append(g)
        m6_nulls.append(nl)

proposal_tests_df = pd.concat([proposal_tests_df, _standardize_tests(pd.concat(m6_tests, ignore_index=True))], ignore_index=True)
proposal_curves_df = pd.concat([proposal_curves_df, _standardize_curves(pd.concat(m6_curves, ignore_index=True))], ignore_index=True)
proposal_gradient_df = pd.concat([proposal_gradient_df, _standardize_gradient(pd.concat(m6_grads, ignore_index=True))], ignore_index=True)
proposal_nulls_df = pd.concat([proposal_nulls_df, _standardize_nulls(pd.concat(m6_nulls, ignore_index=True))], ignore_index=True)
proposal_tests_df = _standardize_tests(proposal_tests_df)
proposal_tests_df.tail()

,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,human_ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
739,persona,proposals,original,whole,human_vs_gemini,coverage,coverage_mesh_terms,False,k_lit=20,1248.0,...,1239.25,permutation,872.000,0.000100,NaN,23,23,10000.0,1.0,M6 uses abstract-derived KNN; BERTopic outlier...
740,persona,proposals,original,whole,human_vs_gpt,coverage,coverage_bertopic_region,False,k_lit=20,10.0,...,10.00,permutation,5.000,0.000500,NaN,23,23,10000.0,1.0,M6 uses abstract-derived KNN; BERTopic outlier...
741,persona,proposals,original,whole,human_vs_gpt,coverage,coverage_mesh_terms,False,k_lit=20,1248.0,...,1239.25,permutation,678.000,0.000100,NaN,23,23,10000.0,1.0,M6 uses abstract-derived KNN; BERTopic outlier...
742,persona,proposals,original,whole,human_vs_pooled_ai,coverage,coverage_bertopic_region,False,k_lit=20,10.0,...,10.00,same_size_subsample,5.301,0.000999,NaN,23,23,1000.0,1.0,M6 uses abstract-derived KNN; BERTopic outlier...
743,persona,proposals,original,whole,human_vs_pooled_ai,coverage,coverage_mesh_terms,False,k_lit=20,1248.0,...,1239.25,same_size_subsample,715.377,0.000999,NaN,23,23,1000.0,1.0,M6 uses abstract-derived KNN; BERTopic outlier...


## Interleaving Statistics (SI)

Descriptive "unique territory" check: for each proposal, the distance to its nearest neighbor in the *other* group, benchmarked against human-to-human spacing (yardstick = 90th percentile of human→nearest-other-human distance). Shares beyond the yardstick near the ~10% by-construction reference mean the groups are interleaved, not partitioned. No inference; written to `facet_interleaving.csv`, drawn as an SI panel by `04`.

In [5]:
inter_rows = []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        _, _, master, idx_cache = _assert_proposal_contract(condition, text_version)
        analysis = load_condition_analysis_inputs(PROJECT_ROOT, condition, text_version=text_version)
        X = di.l2_normalize(np.asarray(analysis.full_embeddings["embeddings"], dtype=float))
        human_rows = master.index[master["source_type"].eq("human")].to_numpy()
        Xh = X[human_rows]
        payloads = {}
        for g in ["Claude", "Gemini", "GPT"]:
            rows_g = master.index[master["source_group"].eq(g)].to_numpy()
            payloads[f"human_vs_{g.lower()}"] = di.interleaving_distances(Xh, X[rows_g])
        # Pooled AI at equal n: distances pooled over 200 cached without-replacement subsamples of 23.
        d_ah_all, d_ha_all, d_hh = [], [], None
        for sample in idx_cache[:200]:
            d_ah, d_hh, d_ha = di.interleaving_distances(Xh, X[np.asarray(sample, dtype=int)])
            d_ah_all.append(d_ah)
            d_ha_all.append(d_ha)
        payloads["human_vs_pooled_ai"] = (np.concatenate(d_ah_all), d_hh, np.concatenate(d_ha_all))
        for comparison, (d_ah, d_hh_c, d_ha) in payloads.items():
            for stat, value in di.interleaving_summary(d_ah, d_hh_c, d_ha).items():
                inter_rows.append({"condition": condition, "task": "proposals", "text_version": text_version,
                                   "field": "whole", "comparison": comparison, "stat": stat, "value": value})
proposal_interleaving_df = pd.DataFrame(inter_rows)
proposal_interleaving_df[proposal_interleaving_df.stat.isin(["share_human_fringe", "share_ai_pocket"])] \
    .pivot_table(index=["condition", "text_version"], columns=["comparison", "stat"], values="value").round(3)


comparison                 human_vs_claude                    human_vs_gemini  \
stat                       share_ai_pocket share_human_fringe share_ai_pocket   
condition     text_version                                                      
baseline      original               0.043              0.217           0.348   
              rephrased              0.087              0.174           0.174   
one_at_a_time original               0.000              0.435           0.000   
              rephrased              0.043              0.130           0.087   
persona       original               0.000              0.087           0.000   
              rephrased              0.087              0.261           0.043   

comparison                                       human_vs_gpt  \
stat                       share_human_fringe share_ai_pocket   
condition     text_version                                      
baseline      original                  0.435           0.043   
              rephrased                 0.348           0.000   
one_at_a_time original                  0.435           0.000   
              rephrased                 0.348           0.000   
persona       original                  0.130           0.000   
              rephrased                 0.261           0.130   

comparison                                    human_vs_pooled_ai  \
stat                       share_human_fringe    share_ai_pocket   
condition     text_version                                         
baseline      original                  0.261              0.146   
              rephrased                 0.217              0.089   
one_at_a_time original                  0.348              0.000   
              rephrased                 0.217              0.043   
persona       original                  0.130              0.000   
              rephrased                 0.217              0.085   

comparison                                     
stat                       share_human_fringe  
condition     text_version                     
baseline      original                  0.244  
              rephrased                 0.223  
one_at_a_time original                  0.305  
              rephrased                 0.184  
persona       original                  0.126  
              rephrased                 0.256

## FDR + Facet Convergence

Benjamini–Hochberg runs once over the full secondary family within each `(task, text_version, field)` — across conditions, comparisons, and metrics, M6 included (spec §1.7). The three pre-registered primaries stay on `p_raw` (their `p_fdr` is NaN by design). The convergence matrix (spec §3.4) correlates per-group metric *values* across all `condition × text_version` cells: the M0 block should read uniformly high (one facet, several views) while off-block cells read lower (facets are non-redundant).

In [6]:
# Family-level FDR (secondaries only; primaries reported on p_raw - spec 1.7).
proposal_tests_df = di.apply_family_fdr(proposal_tests_df)

# Gradient FDR across the whole gradient family per (task, text_version).
if not proposal_gradient_df.empty:
    proposal_gradient_df["p_fdr"] = np.nan
    for _, idx in proposal_gradient_df.groupby(["task", "text_version"]).groups.items():
        idx = list(idx)
        proposal_gradient_df.loc[idx, "p_fdr"] = di.benjamini_hochberg(proposal_gradient_df.loc[idx, "p_raw"])

# Per-group metric-value matrix for the convergence panel (all conditions x versions).
convergence_matrix = pl.build_group_value_matrix(proposal_tests_df)
print(f"Convergence matrix: {convergence_matrix.shape[0]} group-cells x {convergence_matrix.shape[1]} metrics")
n_secondary = int((~proposal_tests_df["is_primary"] & np.isfinite(proposal_tests_df["p_raw"])).sum())
print(f"FDR family size (secondaries with finite p, both text versions): {n_secondary}")
proposal_tests_df.groupby(["facet", "is_primary"])["p_fdr"].apply(lambda s: s.notna().mean()).round(2)

Convergence matrix: 24 group-cells x 12 metrics
FDR family size (secondaries with finite p, both text versions): 510


facet            is_primary
coverage         False         0.75
                 True          0.00
dimensionality   False         1.00
                 True          0.00
displacement     False         0.80
                 True          0.00
evenness         False         1.00
                 True          0.00
lexical_control  False         0.67
richness         False         1.00
                 True          0.00
spread           False         1.00
                 True          0.00
Name: p_fdr, dtype: float64

## Export

Write tidy tables per `{condition}/proposals/{text_version}` and cross-condition copies.

In [7]:
def _write_curves(path, df):
    out = df.copy()
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        if numeric_col in out.columns:
            out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    try:
        out.to_parquet(path, index=False)
    except Exception as exc:
        raise RuntimeError(f"Could not write required parquet {path}: {exc}") from exc


# Fingerprint construction (redesign spec 3.1): one shared, sign-aligned standardized axis.
FINGERPRINT_SPECS = [
    ("spread", "mean_pairwise", "", +1),
    ("richness", "vendi", "q=1", +1),
    ("evenness", "ripley_excess", "r=pooled_q01_q50", -1),
    ("dimensionality", "participation_ratio", "", +1),
    ("coverage", "coverage_geometric", "k=3", +1),
    ("coverage", "coverage_bertopic_region", "k_lit=10", +1),
]

def _z(value, mu, sd, sign):
    return sign * (value - mu) / sd if (np.isfinite(value) and sd > 0) else np.nan

def _fingerprint_rows(condition, text_version):
    tests = proposal_tests_df[(proposal_tests_df.condition == condition)
                              & (proposal_tests_df.text_version == text_version)
                              & proposal_tests_df.field.eq("whole")]
    nulls = proposal_nulls_df[(proposal_nulls_df.condition == condition)
                              & (proposal_nulls_df.text_version == text_version)]
    curves = proposal_curves_df[(proposal_curves_df.condition == condition)
                                & (proposal_curves_df.text_version == text_version)]
    out = []
    for facet, metric, param, sign in FINGERPRINT_SPECS:
        if metric == "coverage_geometric":
            # Coverage null = the human split-half distribution (redesign spec 3.1 exception).
            null_vals = curves.loc[curves.metric.eq("coverage_geometric_split_half"), "y"].dropna().to_numpy(dtype=float)
        else:
            null_vals = nulls.loc[nulls.facet.eq(facet) & nulls.metric.eq(metric) & nulls.param.eq(param), "value"].dropna().to_numpy(dtype=float)
        if null_vals.size < 10:
            continue
        mu, sd = float(np.mean(null_vals)), float(np.std(null_vals))
        rows = tests[tests.facet.eq(facet) & tests.metric.eq(metric) & tests.param.eq(param)]
        if rows.empty:
            continue
        # Human (identical across comparisons; take a per-model row for the jackknife CI).
        h_row = rows[rows.comparison.ne("human_vs_pooled_ai")].iloc[0]
        lo, hi = (h_row["human_ci_lo"], h_row["human_ci_hi"]) if sign > 0 else (h_row["human_ci_hi"], h_row["human_ci_lo"])
        out.append({"condition": condition, "task": "proposals", "text_version": text_version, "field": "whole",
                    "facet": facet, "metric": metric, "param": param, "group": "Human",
                    "value": h_row["human_value"], "z": _z(h_row["human_value"], mu, sd, sign),
                    "z_ci_lo": _z(lo, mu, sd, abs(sign)) * (1 if sign > 0 else -1) if np.isfinite(lo) else np.nan,
                    "z_ci_hi": _z(hi, mu, sd, abs(sign)) * (1 if sign > 0 else -1) if np.isfinite(hi) else np.nan,
                    "sign_aligned": sign, "stars": "", "mode": "z"})
        for comparison, group in [("human_vs_claude", "Claude"), ("human_vs_gemini", "Gemini"),
                                  ("human_vs_gpt", "GPT"), ("human_vs_pooled_ai", "All AI")]:
            r = rows[rows.comparison.eq(comparison)]
            if r.empty:
                continue
            r = r.iloc[0]
            lo, hi = (r["ci_lo"], r["ci_hi"]) if sign > 0 else (r["ci_hi"], r["ci_lo"])
            out.append({"condition": condition, "task": "proposals", "text_version": text_version, "field": "whole",
                        "facet": facet, "metric": metric, "param": param, "group": group,
                        "value": r["ai_value"], "z": _z(r["ai_value"], mu, sd, sign),
                        "z_ci_lo": sign * (lo - mu) / sd if np.isfinite(lo) and sd > 0 else np.nan,
                        "z_ci_hi": sign * (hi - mu) / sd if np.isfinite(hi) and sd > 0 else np.nan,
                        "sign_aligned": sign, "stars": pl.row_stars(r), "mode": "z"})
    fp = pd.DataFrame(out)
    if not fp.empty:
        # Guard against swapped CI arms after sign flips.
        swap = fp["z_ci_lo"] > fp["z_ci_hi"]
        fp.loc[swap, ["z_ci_lo", "z_ci_hi"]] = fp.loc[swap, ["z_ci_hi", "z_ci_lo"]].to_numpy()
    return fp


def _write_cell_outputs(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "proposals" / text_version
    figures_dir = PROJECT_ROOT / "results" / "figures" / condition / "proposals" / text_version
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    tests = proposal_tests_df[(proposal_tests_df.condition == condition) & (proposal_tests_df.text_version == text_version)].copy()
    gradients = proposal_gradient_df[(proposal_gradient_df.condition == condition) & (proposal_gradient_df.text_version == text_version)].copy()
    curves = proposal_curves_df[(proposal_curves_df.condition == condition) & (proposal_curves_df.text_version == text_version)].copy()
    nulls = proposal_nulls_df[(proposal_nulls_df.condition == condition) & (proposal_nulls_df.text_version == text_version)].copy()
    tests.to_csv(tables_dir / "facet_diversity_tests.csv", index=False)
    gradients.to_csv(tables_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(tables_dir / "facet_diversity_curves.parquet", curves)
    _write_curves(tables_dir / "facet_null_reference.parquet", nulls.rename(columns={"draw_idx": "x", "value": "y"}).assign(group="PooledNull", y_lo=np.nan, y_hi=np.nan))
    _fingerprint_rows(condition, text_version).to_csv(tables_dir / "facet_fingerprint.csv", index=False)
    inter = proposal_interleaving_df[(proposal_interleaving_df.condition == condition) & (proposal_interleaving_df.text_version == text_version)]
    inter.to_csv(tables_dir / "facet_interleaving.csv", index=False)
    return tables_dir, figures_dir

written = []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        written.append((condition, text_version, *_write_cell_outputs(condition, text_version)))

for text_version in CONFIG["text_versions"]:
    cross_dir = PROJECT_ROOT / "results" / "tables" / "cross_condition" / "proposals" / text_version
    cross_dir.mkdir(parents=True, exist_ok=True)
    proposal_tests_df[proposal_tests_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_tests.csv", index=False)
    proposal_gradient_df[proposal_gradient_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(cross_dir / "facet_diversity_curves.parquet", proposal_curves_df[proposal_curves_df.text_version.eq(text_version)])

pd.DataFrame(written, columns=["condition", "text_version", "tables_dir", "figures_dir"])

,condition,text_version,tables_dir,figures_dir
0,baseline,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,baseline,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
2,one_at_a_time,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
3,one_at_a_time,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
4,persona,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
5,persona,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...


## Figures

Every required §12.7 view is drawn from the tidy tables/curves just written — box (jackknife/subsample distributions), ridge, effect, profile, scree, envelope (simultaneous), CDF, NN-similarity histogram, density–coverage scatter, rarefaction (k_lit=10), region-occupancy heatmap, MMD² bar with permutation null band and split-half floor, and the two illustration UMAPs. Saved as PNG + PDF. No curve is recomputed in a plotting cell.

In [8]:
def _bool_col(df, col):
    if col not in df.columns:
        return np.zeros(len(df), dtype=bool)
    return df[col].fillna(False).astype(str).str.lower().isin(["true", "1", "1.0"]).to_numpy()


def _emit_umap_figures(condition, text_version, figs, fig_ctx):
    prep = PROJECT_ROOT / "data" / "prepared" / condition / "proposals" / text_version
    coords = np.load(prep / "proposal_umap2d.npy")
    master = pd.read_csv(prep / "proposal_master.csv")
    funded = _bool_col(master, "is_funded_human")
    top = _bool_col(master, "is_top5_ranked_human") | _bool_col(master, "is_top5_AI-ranked")
    pl.plot_group_umap(coords, master["source_group"], figs / "proposal_space_umap",
                       title=f"Proposal-space UMAP (illustration only) · {fig_ctx}",
                       funded_mask=funded, top_mask=top)
    # Literature-anchored map: proposals placed at their nearest literature abstract on the
    # FROZEN literature map (cached coordinates only - no refit, spec 1A.8). The KNN is
    # abstract-derived, keeping markers comparable to literature abstracts.
    lit_coords = np.load(PROJECT_ROOT / "data" / "embeddings" / "literature" / "lit_umap2d.npy")
    knn = np.load(prep / "proposal_to_literature_knn.npz", allow_pickle=True)
    nn0 = np.asarray(knn["neighbor_idx"])[:, 0].astype(int)
    cmap = plt.get_cmap("tab20")
    bg_colors = [cmap(int(t) % 20) if int(t) >= 0 else (0.85, 0.85, 0.85, 1.0) for t in TOPIC_BY_ROW]
    pl.plot_group_umap(lit_coords[nn0], master["source_group"], figs / "literature_anchored_umap",
                       title=f"Literature-anchored UMAP (illustration only) · {fig_ctx}",
                       funded_mask=funded, top_mask=top,
                       xlabel="Literature UMAP Dim 1", ylabel="Literature UMAP Dim 2",
                       background=(lit_coords, bg_colors))


def emit_required_figures(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "proposals" / text_version
    figs = PROJECT_ROOT / "results" / "figures" / condition / "proposals" / text_version
    tests = pd.read_csv(tables_dir / "facet_diversity_tests.csv")
    curves = pd.read_parquet(tables_dir / "facet_diversity_curves.parquet")
    tests["param"] = tests["param"].fillna("")
    curves["param"] = curves["param"].fillna("")
    tests = tests[tests["field"].eq("whole")]
    curves = curves[curves["field"].eq("whole")]
    fig_ctx = f"{condition} · proposals/{text_version}"

    # Fingerprint (redesign spec 3.1): every facet on one sign-aligned standardized axis.
    fingerprint = pd.read_csv(tables_dir / "facet_fingerprint.csv")
    fingerprint["param"] = fingerprint["param"].fillna("")
    pl.plot_fingerprint(fingerprint, figs / "facet_fingerprint", mode="z",
                        title=f"Diversity fingerprint — all facets, one axis · {fig_ctx}")

    # Spread (M0)
    pl.plot_setlevel_box(tests, curves, figs / "spread_mean_pairwise_box", facet="spread", metric="mean_pairwise",
                         param="", title=f"Spread — mean_pairwise · {fig_ctx}", ylabel="Pairwise cosine distance (mean)")
    pl.plot_pairwise_ridge(curves, figs / "spread_mean_pairwise_ridge",
                           title=f"Spread — mean_pairwise, full distributions · {fig_ctx}")
    pl.plot_effect_ratio(tests, figs / "spread_mean_pairwise_effect", facet="spread", metric="mean_pairwise",
                         param="", title=f"Spread — mean_pairwise · {fig_ctx}", xlabel="Mean pairwise distance: Human ÷ AI")
    pl.plot_pooled_subsample_hist(curves, tests, figs / "spread_mean_pairwise_pooled_hist", metric="mean_pairwise",
                                  param="", title=f"Spread — mean_pairwise, pooled Human vs All-AI · {fig_ctx}",
                                  xlabel="Mean pairwise cosine distance")
    pl.plot_convergent_spread_box(curves, figs / "spread_convergent_box",
                                  title=f"Spread — convergent metrics · {fig_ctx}")

    # Richness (M1)
    pl.plot_vendi_profile(curves, figs / "richness_vendi_profile", title=f"Richness — Vendi profile · {fig_ctx}")
    pl.plot_scree(curves, figs / "richness_vendi_scree", metric="kernel_eigen_scree",
                  title=f"Richness — Vendi eigenvalue scree (diagnostic) · {fig_ctx}", xlabel="eigen-index",
                  ylabel="normalized kernel eigenvalue (cliff = few dominant modes = LESS diverse)",
                  log_y=True, badge="↓ steeper cliff = less diverse")
    pl.plot_setlevel_box(tests, curves, figs / "richness_vendi_box", facet="richness", metric="vendi", param="q=1",
                         title=f"Richness — Vendi VS₁ · {fig_ctx}", ylabel="Effective number of distinct proposals (VS₁)")
    pl.plot_effect_ratio(tests, figs / "richness_vendi_effect", facet="richness", metric="vendi", param="q=1",
                         title=f"Richness — Vendi VS₁ · {fig_ctx}", xlabel="Effective distinct proposals: Human ÷ AI")

    # Evenness (M4 + vendi_slope) — re-oriented per redesign spec 2.
    pl.plot_ripley_envelope(curves, figs / "evenness_ripley_excess_envelope",
                            title=f"Evenness — vs same-size null · {fig_ctx}")
    pl.plot_g_cdf(curves, figs / "evenness_g_function_cdf", title=f"Evenness — twin-free fraction (1−G) · {fig_ctx}")
    pl.plot_nn_distance_hist(curves, figs / "evenness_nn_similarity_hist",
                             title=f"Evenness — NN distances · {fig_ctx}")
    pl.plot_setlevel_box(tests, curves, figs / "evenness_vendi_slope_box", facet="evenness", metric="vendi_slope",
                         param="q=0..2", title=f"Evenness — vendi_slope · {fig_ctx}",
                         ylabel="Vendi profile drop (VS₀−VS₂)/VS₀; higher = less even")

    # Dimensionality (M3) — residual-variation orientation per redesign spec 2.
    pl.plot_scree(curves, figs / "dimensionality_participation_ratio_scree", metric="participation_ratio_scree",
                  title=f"Dimensionality — residual variation · {fig_ctx}", xlabel="principal component index",
                  ylabel="variance remaining beyond first x components", mark_90=True,
                  residual=True, badge="↑ higher-dimensional")
    pl.plot_setlevel_box(tests, curves, figs / "dimensionality_participation_ratio_box", facet="dimensionality",
                         metric="participation_ratio", param="",
                         title=f"Dimensionality — participation_ratio · {fig_ctx}", ylabel="Participation ratio")
    pl.plot_effect_ratio(tests, figs / "dimensionality_participation_ratio_effect", facet="dimensionality",
                         metric="participation_ratio", param="",
                         title=f"Dimensionality — participation_ratio · {fig_ctx}",
                         xlabel="Effective dimensionality: Human ÷ AI")

    # Coverage, geometric (M2)
    pl.plot_coverage_scatter(tests, curves, figs / "coverage_geometric_scatter",
                             title=f"Coverage — geometric, density–coverage · {fig_ctx}")
    pl.plot_coverage_box(tests, curves, figs / "coverage_geometric_box", title=f"Coverage — geometric · {fig_ctx}")
    pl.plot_coverage_effect(tests, figs / "coverage_geometric_effect", title=f"Coverage — geometric · {fig_ctx}")

    # Coverage, domain (M6)
    pl.plot_rarefaction(curves, figs / "coverage_bertopic_region_rarefaction",
                        metric="coverage_bertopic_region_rarefaction",
                        title=f"Coverage — BERTopic regions, rarefaction · {fig_ctx}",
                        ylabel="distinct literature regions touched")
    pl.plot_rarefaction(curves, figs / "coverage_mesh_terms_rarefaction", metric="coverage_mesh_terms_rarefaction",
                        title=f"Coverage — unique MeSH descriptors, rarefaction · {fig_ctx}",
                        ylabel="Unique MeSH descriptors")
    occ = curves[curves["metric"].eq("region_occupancy") & curves["param"].eq("k_lit=10")]
    if not occ.empty:
        matrix = occ.pivot_table(index="x", columns="group", values="y", aggfunc="first")
        matrix = matrix[[c for c in ["Human", "Claude", "Gemini", "GPT"] if c in matrix.columns]]
        matrix.index = [TOPIC_LABELS.get(int(i), f"region {int(i)}") for i in matrix.index]
        matrix = matrix.sort_values("Human", ascending=False)
    else:
        matrix = pd.DataFrame()
    pl.plot_occupancy_heatmap(matrix, figs / "coverage_bertopic_region_heatmap",
                              title=f"Coverage — region occupancy (k_lit=10) · {fig_ctx}")

    # Displacement (M5 — quarantined directional check, redesign spec 1.2)
    pl.plot_mmd_bar(tests, figs / "displacement_mmd2_bar", title=f"Displacement — MMD² · {fig_ctx}")

    # Illustration UMAPs (never metric inputs)
    _emit_umap_figures(condition, text_version, figs, fig_ctx)

    # Convergence panel (spec 3.4): computed over ALL cells, written under each cell path (spec 12.6).
    pl.plot_facet_convergence(convergence_matrix, figs / "_convergence" / "facet_convergence_heatmap",
                              title=f"Facet convergence — Spearman over per-group values, all conditions × text versions\n(shown under {fig_ctx})")


if WRITE_FIGURES:
    for condition in CONFIG["conditions"]:
        for text_version in CONFIG["text_versions"]:
            emit_required_figures(condition, text_version)
print("Proposal facet figures complete.")

Proposal facet figures complete.


## Cross-condition Ratios

Persona-persistence panels (spec §10.1.4, §12.6): the AI÷Human diversity-retained ratio per condition, per model, for each ratio-valued primary metric. `ripley_excess` is excluded here — its effect size is an envelope area, not a ratio (§1A.9); its cross-condition story is carried by the envelope figures.

In [9]:
CROSS_SPECS = [
    ("spread", "mean_pairwise", "", "Mean pairwise distance"),
    ("richness", "vendi", "q=1", "Vendi VS₁"),
    ("coverage", "coverage_geometric", "k=3", "Geometric coverage"),
    ("coverage", "coverage_bertopic_region", "k_lit=10", "Literature regions covered"),
    ("dimensionality", "participation_ratio", "", "Participation ratio"),
]

if WRITE_FIGURES:
    for text_version in CONFIG["text_versions"]:
        ccdir = PROJECT_ROOT / "results" / "figures" / "cross_condition" / "proposals" / text_version
        for facet, metric, param, label in CROSS_SPECS:
            pl.plot_cross_condition_ratio(
                proposal_tests_df, ccdir / f"{facet}_{metric}_ratio_by_condition",
                facet=facet, metric=metric, param=param, task="proposals", text_version=text_version,
                title=f"{facet.title()} — {label} · AI÷Human by condition · proposals/{text_version}")
print("Cross-condition proposal figures complete.")

Cross-condition proposal figures complete.
